In [1]:
%load_ext autoreload
%autoreload 2
%cd ../..

/gpfs/data/schambralab/quantitativeRehabilitation/__lab_member_homes/victor/cvfm4rehab


In [2]:
from postprocess.ia.eval import (
    answers_for_model,
    fm_scores_for_model,
    metrics_for_model,
    metrics_for_models
)

def pivot_fm_scores(df):
    df_long = df.melt(
        id_vars=['patient', 'fm_item'],
        value_vars=['pred_score', 'gt_score'],
        var_name='score_type',
        value_name='score'
    )
    df_long['col'] = df_long['score_type'] + "_" + df_long['patient']
    out = df_long.pivot(index="fm_item", columns="col", values="score")
    return out


def pivot_answers(df):
    melted_df = df.melt(
        id_vars=['patient', 'qid'],
        value_vars=['answer'],
        var_name='answer_type',
    )
    melted_df.drop(columns=['answer_type'], inplace=True)
    melted_df.rename({'value': 'answer'}, axis=1, inplace=True)
    out = (
        melted_df
        .pivot(index="qid", columns="patient", values="answer")
        .add_suffix("_answer")
    )
    return out

In [3]:
from data.utils_strokerehab import DataPaths

In [4]:
MODEL = "qwen2_5_vl_7b"
PATIENT_ORDER = ["C00011", "S0005", "S0001", "S00021"]

In [8]:
answers_for_model(MODEL, tasks=("strokerehab_ia3_3_30","strokerehab_ia3_31_33"))

,patient,qid,answer
0,C00011,3,"In the top video, the patient raises their sho..."
1,C00011,4,"In the top video, the patient's shoulder does ..."
2,C00011,5,"In the top video, the patient successfully rai..."
3,C00011,6,"In the top video, the patient's arm reaches sh..."
4,C00011,7,"In the top video, the patient's elbow flexion ..."
...,...,...,...
127,S0005,96,1
128,S0005,97,5.333
129,S0005,98,5.333
130,S0005,99,6.2


In [9]:
import pandas as pd
pd.set_option('display.max_colwidth', None)
pivot_table = pivot_answers(answers_for_model(MODEL, tasks=("strokerehab_ia3_3_30","strokerehab_ia3_31_33"), drop_parsed=False))

In [10]:
qdf = pd.read_csv("./data/ia/fm_item_questions3.csv")
qdf.head()
qdf.drop(columns=["question_type", "sampling", "binary_no_score", "binary_yes_score"], inplace=True)

In [11]:
df = pd.merge(qdf, pivot_table, left_on="qid", right_index=True, how="inner")
df.set_index("qid")[['C00011_answer', 'S0005_answer', 'S0001_answer', 'S00021_answer', 'fm_video', 'question']].to_csv('scrap.csv', index=False)

In [12]:
fm_scores = pivot_fm_scores(fm_scores_for_model(
    MODEL,
    tasks=("strokerehab_ia3_3_30","strokerehab_ia3_31_33"),
    questions_csv_path=DataPaths.IA_QUESTIONS_PATH3,
    gt_csv_path=DataPaths.IA_SCORES_PATH
))[['gt_score_' + p for p in PATIENT_ORDER] + ['pred_score_' + p for p in PATIENT_ORDER]]
fm_scores

col,gt_score_C00011,gt_score_S0005,gt_score_S0001,gt_score_S00021,pred_score_C00011,pred_score_S0005,pred_score_S0001,pred_score_S00021
fm_item,,,,,,,,
3,2,2,2,1,1,1,1,1
4,2,2,2,0,1,1,1,1
5,2,2,2,1,2,2,2,2
6,2,2,2,0,1,2,0,2
7,2,2,2,0,1,1,0,1
8,2,2,1,0,1,2,1,2
9,2,2,2,0,0,0,1,0
10,2,2,1,0,1,1,2,2
11,2,2,2,0,2,0,2,2


In [13]:
fm_scores.sum(axis=0)

col
gt_score_C00011      62
gt_score_S0005       61
gt_score_S0001       38
gt_score_S00021       4
pred_score_C00011    39
pred_score_S0005     33
pred_score_S0001     36
pred_score_S00021    35
dtype: Int64

In [14]:
metrics_for_model(
    MODEL,
    tasks=("strokerehab_ia3_3_30","strokerehab_ia3_31_33"),
    questions_csv_path=DataPaths.IA_QUESTIONS_PATH3,
    gt_csv_path=DataPaths.IA_SCORES_PATH
)

{'accuracy': 0.2833333333333333, 'apd': 25.5, 'mtsd': 20.5}